# Explore External MWE Datasets: DiMSUM + PARSEME

This notebook helps you inspect the two newly added datasets and test phrase coverage.

Datasets:
- DiMSUM (SemEval 2016 Task 10)
- PARSEME Shared Task (multi-version, multi-language)


In [ ]:
from __future__ import annotations

from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
import re

try:
    import pandas as pd
except Exception:
    pd = None


In [ ]:
cwd = Path.cwd().resolve()
ROOT = cwd if (cwd / 'app').exists() else cwd.parent

DIMSUM_ROOT = ROOT / 'data' / 'external' / 'dimsum' / 'dimsum16-dimsum-data-cd92971'
PARSEME_ROOT = ROOT / 'data' / 'external' / 'parseme' / 'sharedtask-data-master'

print('DiMSUM root:', DIMSUM_ROOT, '| exists =', DIMSUM_ROOT.exists())
print('PARSEME root:', PARSEME_ROOT, '| exists =', PARSEME_ROOT.exists())


## DiMSUM loader

In [ ]:
def read_dimsum(path: Path):
    # Expected columns: ID TOKEN LEMMA UPOS MWE_TAG MWE_PARENT ... SENT_ID
    sentences = []
    sent = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n')
            if not line:
                if sent:
                    sentences.append(sent)
                    sent = []
                continue
            parts = line.split('\t')
            if len(parts) < 9:
                continue
            sent.append({
                'id': parts[0],
                'token': parts[1],
                'lemma': parts[2],
                'upos': parts[3],
                'mwe_tag': parts[4],
                'mwe_parent': parts[5],
                'supersense': parts[7],
                'sent_id': parts[8],
            })
    if sent:
        sentences.append(sent)
    return sentences


dimsum_train = read_dimsum(DIMSUM_ROOT / 'dimsum16.train')
dimsum_test_blind = read_dimsum(DIMSUM_ROOT / 'dimsum16.test.blind')

print('DiMSUM train sentences:', len(dimsum_train))
print('DiMSUM blind test sentences:', len(dimsum_test_blind))


In [ ]:
def dimsum_stats(sentences):
    token_count = 0
    tag_counts = Counter()
    mwe_token_count = 0
    for sent in sentences:
        for tok in sent:
            token_count += 1
            tag = tok['mwe_tag']
            tag_counts[tag] += 1
            if tag in {'B','I'}:
                mwe_token_count += 1
    return {
        'sentences': len(sentences),
        'tokens': token_count,
        'mwe_tokens': mwe_token_count,
        'mwe_token_ratio': round(mwe_token_count / token_count, 4) if token_count else 0.0,
        'tag_counts': dict(tag_counts),
    }

stats_train = dimsum_stats(dimsum_train)
stats_blind = dimsum_stats(dimsum_test_blind)
stats_train, stats_blind


In [ ]:
def dimsum_sentence_text(sent):
    return ' '.join(tok['token'] for tok in sent)


def search_dimsum_phrase(phrase: str, sentences, limit: int = 10):
    q = phrase.lower().strip()
    out = []
    for sent in sentences:
        text = dimsum_sentence_text(sent)
        if q in text.lower():
            out.append(text)
            if len(out) >= limit:
                break
    return out

# Example
search_dimsum_phrase('get going', dimsum_train, limit=5)


## PARSEME loader (Cupt)

In [ ]:
cupt_files = sorted(PARSEME_ROOT.rglob('*.cupt'))
print('Total .cupt files:', len(cupt_files))

version_counter = Counter()
lang_counter = Counter()
for p in cupt_files:
    rel = p.relative_to(PARSEME_ROOT)
    parts = rel.parts
    if len(parts) >= 2:
        version_counter[parts[0]] += 1
        lang_counter[parts[1]] += 1

print('Files by version:', dict(version_counter))
print('Top languages by file count:', lang_counter.most_common(15))


In [ ]:
@dataclass
class ParsemeMWE:
    mwe_id: str
    category: str
    tokens: list[str]
    lemmas: list[str]
    sentence_text: str


def parse_cupt_sentences(path: Path):
    # Reads token rows and sentence metadata from .cupt files
    sentences = []
    sent_tokens = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n')
            if not line:
                if sent_tokens:
                    sentences.append(sent_tokens)
                    sent_tokens = []
                continue
            if line.startswith('#'):
                continue
            cols = line.split('\t')
            if len(cols) < 11:
                continue
            tok_id = cols[0]
            if '-' in tok_id or '.' in tok_id:
                continue
            sent_tokens.append({
                'id': int(tok_id),
                'form': cols[1],
                'lemma': cols[2],
                'mwe': cols[10],
            })
    if sent_tokens:
        sentences.append(sent_tokens)
    return sentences


def extract_parseme_mwes(sentences):
    all_mwes = []
    for sent in sentences:
        by_id = defaultdict(lambda: {'category': '', 'tokens': [], 'lemmas': []})
        for tok in sent:
            tag = tok['mwe']
            if tag in {'*', '_', ''}:
                continue
            parts = tag.split(';')
            for part in parts:
                part = part.strip()
                if not part:
                    continue
                if ':' in part:
                    mwe_id, cat = part.split(':', 1)
                    by_id[mwe_id]['category'] = cat
                    by_id[mwe_id]['tokens'].append(tok['form'])
                    by_id[mwe_id]['lemmas'].append(tok['lemma'])
                else:
                    by_id[part]['tokens'].append(tok['form'])
                    by_id[part]['lemmas'].append(tok['lemma'])
        sent_text = ' '.join(t['form'] for t in sent)
        for mwe_id, data in by_id.items():
            all_mwes.append(ParsemeMWE(
                mwe_id=mwe_id,
                category=data['category'],
                tokens=data['tokens'],
                lemmas=data['lemmas'],
                sentence_text=sent_text,
            ))
    return all_mwes


In [ ]:
# Use English trial set for quick experimentation
en_trial_train = PARSEME_ROOT / '1.2' / 'trial' / 'EN-trial.train.cupt'
en_trial_sents = parse_cupt_sentences(en_trial_train)
en_trial_mwes = extract_parseme_mwes(en_trial_sents)

print('EN-trial sentences:', len(en_trial_sents))
print('EN-trial extracted MWEs:', len(en_trial_mwes))

cat_counts = Counter(m.category or 'UNKNOWN' for m in en_trial_mwes)
cat_counts


In [ ]:
def search_parseme_mwes(query: str, mwes, limit: int = 10):
    q = query.lower().strip()
    rows = []
    for m in mwes:
        phrase = ' '.join(m.tokens).lower()
        lemma_phrase = ' '.join(m.lemmas).lower()
        if q in phrase or q in lemma_phrase:
            rows.append({
                'category': m.category,
                'surface': ' '.join(m.tokens),
                'lemmas': ' '.join(m.lemmas),
                'sentence': m.sentence_text,
            })
            if len(rows) >= limit:
                break
    if pd is not None:
        return pd.DataFrame(rows)
    return rows


# Examples
search_parseme_mwes('get rid', en_trial_mwes, limit=5)


In [ ]:
# Coverage check for target expressions in both datasets
targets = ['get going', 'screw with', 'get to know', 'plan ahead']

dimsum_texts = [dimsum_sentence_text(s).lower() for s in dimsum_train]
parseme_surfaces = [' '.join(m.tokens).lower() for m in en_trial_mwes]
parseme_lemmas = [' '.join(m.lemmas).lower() for m in en_trial_mwes]

rows = []
for t in targets:
    rows.append({
        'expression': t,
        'in_dimsum_train_sentence_text': any(t in x for x in dimsum_texts),
        'in_parseme_en_trial_surface_mwe': any(t in x for x in parseme_surfaces),
        'in_parseme_en_trial_lemma_mwe': any(t in x for x in parseme_lemmas),
    })

if pd is not None:
    pd.DataFrame(rows)
else:
    rows


## Built Runtime Supplement
Use `app/scripts/build_external_mwe_supplement.py` output to see what was integrated into runtime lookup.

In [ ]:
supp_path = ROOT / 'data' / 'lexicons' / 'phrasal_verbs_external_mwe.json'
supp = json.loads(supp_path.read_text(encoding='utf-8')) if supp_path.exists() else {}
print('supplement path:', supp_path)
print('supplement entries:', len(supp))
for q in ['get going', 'screw with', 'get to know', 'plan ahead']:
    print(q, q in supp)
